# Depth 2 — Robustness and sensitivity

**Setup**: The 10-seed cross-modal results in the main notebook used `K_PCA=256`,
`K_PLS=64`, and (mostly) PLS as the latent regressor. Reviewers will ask: are the
asymmetry ratios artifacts of those choices?

This notebook does the three table-stakes robustness checks. None of them are
discovery — they all *defend* findings already in the main notebook.

## Sections

| Section | Question | Cost |
|---|---|---|
| A | K_PCA / K_PLS sweep — does the asymmetry ratio swing with dimensionality? | ~20 min single-seed |
| B | Non-PLS estimator (BayesianRidge per component) — does the asymmetry survive a different inductive bias? | ~10 min (uses helper) |
| C | `combined_pred_SC` sibling-AUC follow-up — does the diagnostic verdict from STEP 8.4 in the main notebook hold under a hyperparameter perturbation? | ~5 min |

Output: `../model_overviews/results/local_results/further_exploration/depth2_robustness_sensitivity/`


In [ ]:
# ============= SETUP =============
# Single source of truth for data + helpers is _setup.py next to this notebook.
# All notebooks in this directory share the same seed-0 split + closed-form helpers.
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from _setup import *  # load_seed_split, pca_pls_predict, combined_predict, br_per_component_predict,
                       # fit_basis_ols, full_panel_eval, pair_indices_by_relation,
                       # demeaned_cosine_pair_sim, extract_pair_sims, auc_vs_unrelated,
                       # results_dir; also np, pd, torch, PCA, BayesianRidge, LinearRegression

split = load_seed_split(seed=0)
base       = split["base"]
train_idx  = split["train_idx"]
test_idx   = split["test_idx"]
FC_tr, FC_te = split["FC_train"], split["FC_test"]
SC_tr, SC_te = split["SC_train"], split["SC_test"]
bv_tr, bv_te = split["bv_train"], split["bv_test"]
demo_tr, demo_te = split["demo_train"], split["demo_test"]
bvdemo_tr, bvdemo_te = split["bvdemo_train"], split["bvdemo_test"]
SC_train_mean = SC_tr.mean(axis=0)
FC_train_mean = FC_tr.mean(axis=0)

print(f"Setup OK. train={len(train_idx)}  test={len(test_idx)}  parc={PARCELLATION}")
print(f"  FC shape: {FC_tr.shape},  SC shape: {SC_tr.shape}")
print(f"  bv: {bv_tr.shape[1]}-dim, demo: {demo_tr.shape[1]}-dim, bv+demo: {bvdemo_tr.shape[1]}-dim")


---

## Section A — K_PCA / K_PLS sweep

Single seed 0, basis = `none` (cross-modal only), both directions. Grid:
`K_PCA ∈ {64, 128, 256, 512}` × `K_PLS ∈ {16, 32, 64, 128}`.

For each cell of the grid, record the demeaned-pearson R² for FC→SC and SC→FC,
and compute the asymmetry ratio. A flat ratio across the grid is the desired outcome.
A ratio that swings from 1.0 to 2.0 means the asymmetry partly reflects the K choice.


In [ ]:
# ===== Section A: K_PCA × K_PLS sweep, single seed, no basis =====
K_PCAs = [64, 128, 256, 512]
K_PLSs = [16, 32, 64, 128]

sweep_rows = []
for k_pca in K_PCAs:
    for k_pls in K_PLSs:
        if k_pls > k_pca:
            continue  # PLS components can't exceed PCA components
        pred_fs = pca_pls_predict(FC_tr, FC_te, SC_tr, k_src=k_pca, k_tgt=k_pca, k_pls=k_pls)
        panel_fs = full_panel_eval(pred_fs, SC_te, SC_train_mean)
        pred_sf = pca_pls_predict(SC_tr, SC_te, FC_tr, k_src=k_pca, k_tgt=k_pca, k_pls=k_pls)
        panel_sf = full_panel_eval(pred_sf, FC_te, FC_train_mean)
        ratio = panel_fs["demeaned_pearson"] / max(panel_sf["demeaned_pearson"], 1e-9)
        sweep_rows.append({
            "K_PCA":     k_pca,
            "K_PLS":     k_pls,
            "FC_to_SC_dp": panel_fs["demeaned_pearson"],
            "SC_to_FC_dp": panel_sf["demeaned_pearson"],
            "ratio":      ratio,
        })
        print(f"  K_PCA={k_pca:3d} K_PLS={k_pls:3d}  "
              f"FC->SC={panel_fs['demeaned_pearson']:.4f}  "
              f"SC->FC={panel_sf['demeaned_pearson']:.4f}  "
              f"ratio={ratio:.3f}")

sweep_df = pd.DataFrame(sweep_rows)
out_dir2 = results_dir("depth2_robustness_sensitivity")
sweep_df.to_csv(out_dir2 / "k_sweep.csv", index=False)
print(f"\nSaved -> {out_dir2 / 'k_sweep.csv'}")

# Pivot for an at-a-glance grid.
pivot = sweep_df.pivot(index="K_PCA", columns="K_PLS", values="ratio")
print("\nAsymmetry ratio (FC->SC / SC->FC) grid:")
print(pivot.to_string(float_format=lambda x: f"{x:.3f}"))

ratio_min, ratio_max = sweep_df["ratio"].min(), sweep_df["ratio"].max()
print(f"\nRatio range: [{ratio_min:.3f}, {ratio_max:.3f}]  "
      f"(at main notebook's K_PCA=256 K_PLS=64, "
      f"ratio={sweep_df.query('K_PCA==256 and K_PLS==64')['ratio'].iloc[0]:.3f})")
print("Verdict template:")
print("  - Range tight (max - min < 0.2)  -> ROBUST to K choice")
print("  - Range wide (max - min > 0.4)   -> SENSITIVE; report K-sweep in paper")


---

## Section B — Non-PLS estimator robustness

Single seed 0, basis = `none`. Swap PLS for `br_per_component_predict` (BayesianRidge
per target PCA component). Different inductive bias: PLS uses a shared latent space,
BR shrinks per-component independently.

If asymmetry holds under both estimators, the effect is not an artifact of PLS's
particular regularization.

The main notebook's STEP 11 already ran 10-seed PLS. This is a single-seed BR baseline
on top — Tier 2.1 in `../tier_extensions.ipynb` does the full 10-seed BR for paper-grade.


In [ ]:
# ===== Section B: BR-per-component on no-basis cross-modal =====
print("FC -> SC (BR per component)...")
pred_fs_br = br_per_component_predict(FC_tr, FC_te, SC_tr)
panel_fs_br = full_panel_eval(pred_fs_br, SC_te, SC_train_mean)
print(f"  demeaned_pearson = {panel_fs_br['demeaned_pearson']:.4f}")

print("SC -> FC (BR per component)...")
pred_sf_br = br_per_component_predict(SC_tr, SC_te, FC_tr)
panel_sf_br = full_panel_eval(pred_sf_br, FC_te, FC_train_mean)
print(f"  demeaned_pearson = {panel_sf_br['demeaned_pearson']:.4f}")

ratio_br = panel_fs_br["demeaned_pearson"] / max(panel_sf_br["demeaned_pearson"], 1e-9)
print(f"\nBR asymmetry ratio = {ratio_br:.3f}")

# Compare to PLS at same K.
pred_fs_pls = pca_pls_predict(FC_tr, FC_te, SC_tr)
panel_fs_pls = full_panel_eval(pred_fs_pls, SC_te, SC_train_mean)
pred_sf_pls = pca_pls_predict(SC_tr, SC_te, FC_tr)
panel_sf_pls = full_panel_eval(pred_sf_pls, FC_te, FC_train_mean)
ratio_pls = panel_fs_pls["demeaned_pearson"] / max(panel_sf_pls["demeaned_pearson"], 1e-9)

estimator_df = pd.DataFrame([
    {"estimator": "PLS", "FC->SC_dp": panel_fs_pls["demeaned_pearson"],
     "SC->FC_dp": panel_sf_pls["demeaned_pearson"], "ratio": ratio_pls},
    {"estimator": "BR",  "FC->SC_dp": panel_fs_br["demeaned_pearson"],
     "SC->FC_dp": panel_sf_br["demeaned_pearson"], "ratio": ratio_br},
])
print("\nEstimator comparison:")
print(estimator_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
estimator_df.to_csv(out_dir2 / "estimator_comparison.csv", index=False)
print(f"\nSaved -> {out_dir2 / 'estimator_comparison.csv'}")


---

## Section C — `combined_pred_SC` sibling-AUC follow-up

The STEP 8.4 diagnostic in the main notebook produced one of three verdicts on the
sibling-AUC collapse (0.505 vs 0.680 for `pred_SC_raw`):
  - REAL (Caveat a): bv+demo content dominates BR loss, drowning the within-family signal.
  - BUG (Caveat b): a code path issue in `_combined_predict`.
  - BORDERLINE: BR over-regularization.

This section does ONE hyperparameter perturbation suggested by each verdict, to check
whether the AUC moves in the expected direction. **Edit the chosen branch below based
on the diagnostic's verdict** — leaving all three for now since the diagnostic hasn't
been run yet at the time this notebook was scaffolded.


In [ ]:
# ===== Section C: combined_pred_SC perturbation follow-up =====
rng = np.random.default_rng(42)
pairs = pair_indices_by_relation(base.metadata_df, test_idx, rng, age_tol_yrs=3.0)


def sib_auc_of(matrix):
    S = demeaned_cosine_pair_sim(matrix, SC_train_mean)
    sims = extract_pair_sims(S, pairs)
    return auc_vs_unrelated(sims).get("sibling", float("nan"))


# Baseline (reproduce the cached number).
pred_raw  = pca_pls_predict(FC_tr, FC_te, SC_tr)
pred_comb = combined_predict(FC_tr, FC_te, SC_tr, bvdemo_tr, bvdemo_te)
print(f"BASELINE (single seed 0):")
print(f"  pred_SC_raw       sib AUC = {sib_auc_of(pred_raw):.4f}   (cached: 0.680)")
print(f"  combined_pred_SC  sib AUC = {sib_auc_of(pred_comb):.4f}   (cached: 0.505)")

# Perturbation 1: smaller K_PCA(target) -> fewer BR fits, less aggregate shrinkage.
print("\nPerturbation 1: K_PCA(target) reduced 256 -> 128 in combined_predict.")
pred_comb_k128 = combined_predict(FC_tr, FC_te, SC_tr, bvdemo_tr, bvdemo_te, k_tgt=128)
print(f"  sib AUC = {sib_auc_of(pred_comb_k128):.4f}")

# Perturbation 2: larger K_PCA(target) -> more components, finer detail.
print("\nPerturbation 2: K_PCA(target) increased 256 -> 384 in combined_predict.")
pred_comb_k384 = combined_predict(FC_tr, FC_te, SC_tr, bvdemo_tr, bvdemo_te, k_tgt=384)
print(f"  sib AUC = {sib_auc_of(pred_comb_k384):.4f}")

# Perturbation 3: weakened BR prior (BR max_iter raised — proxy for less-aggressive shrinkage
# on this dataset). Not a real prior change but a cheap sanity-check on convergence.
print("\nPerturbation 3: BR max_iter raised 300 -> 1000.")
pred_comb_mi = combined_predict(FC_tr, FC_te, SC_tr, bvdemo_tr, bvdemo_te, max_iter=1000)
print(f"  sib AUC = {sib_auc_of(pred_comb_mi):.4f}")

# Save.
follow_df = pd.DataFrame([
    {"variant": "baseline_K256",         "sib_auc": sib_auc_of(pred_comb)},
    {"variant": "K_tgt=128",             "sib_auc": sib_auc_of(pred_comb_k128)},
    {"variant": "K_tgt=384",             "sib_auc": sib_auc_of(pred_comb_k384)},
    {"variant": "max_iter=1000",         "sib_auc": sib_auc_of(pred_comb_mi)},
    {"variant": "pred_SC_raw (reference)", "sib_auc": sib_auc_of(pred_raw)},
])
follow_df.to_csv(out_dir2 / "combined_followup.csv", index=False)
print(f"\nSaved -> {out_dir2 / 'combined_followup.csv'}")
print("\nVerdict template:")
print("  - If any perturbation lifts sib AUC > 0.6  -> verdict was BORDERLINE/BUG; investigate further")
print("  - If all perturbations stay ~0.50-0.55     -> verdict was REAL (Caveat a)")
